In [18]:
import numpy as np
from sympy import *

init_session()

IPython console for SymPy 1.13.1 (Python 3.12.6-64-bit) (ground types: python)

These commands were executed:
>>> from sympy import *
>>> x, y, z, t = symbols('x y z t')
>>> k, m, n = symbols('k m n', integer=True)
>>> f, g, h = symbols('f g h', cls=Function)
>>> init_printing()

Documentation can be found at https://docs.sympy.org/1.13.1/



In [25]:
def get_transform_mat_from_dh(a, alpha, d, theta):
    return Matrix([[cos(theta), -sin(theta)*cos(alpha),  sin(theta)*sin(alpha), a*cos(theta)],
            [sin(theta),  cos(theta)*cos(alpha), -cos(theta)*sin(alpha), a*sin(theta)],
            [0,        sin(alpha),       cos(alpha),       d],
            [0,             0,            0,       1]])


In [156]:
def load_dh_params(fname='DH_PARAMS.yaml',n=6):
    import yaml
    
    with open(fname, 'r') as f:
        data = yaml.load(f, Loader=yaml.SafeLoader)
    dh_params = []
    links  = list(data.keys())[:n]
    
    for i in links:
        dh_params.append([data[i]['a'],data[i]['alpha'],data[i]['d'],data[i]['theta']])
        
    return dh_params
    
dh_params = load_dh_params()
dh_params = np.round(dh_params,3)
dh_params

array([[-0.   ,  0.   , -0.119,  0.   ],
       [ 0.   ,  0.054, -0.22 ,  0.   ],
       [-0.   ,  0.036, -0.42 ,  0.   ],
       [ 0.   , -0.017, -0.506,  0.   ],
       [ 0.02 , -0.017, -0.563,  0.   ],
       [ 0.   , -0.007, -0.611,  0.   ]])

In [192]:
thetas = symbols('theta1:7')
thetas


(θ₁, θ₂, θ₃, θ₄, θ₅, θ₆)

In [158]:
T = []
n = 6

for i in range(n):
    T.append(get_transform_mat_from_dh(dh_params[i][0],dh_params[i][1],dh_params[i][2],theta[i]))

When the last three axes intersect, the origins of link frames (4), {5}, and {6} are all located at this point of intersection. This point is given in base coordinates as

In [159]:
x,y,z,theta,beta,gamma = symbols('x y z theta beta gamma')
Ps = Matrix([[x],[y],[z],[1]])
Ps

⎡x⎤
⎢ ⎥
⎢y⎥
⎢ ⎥
⎢z⎥
⎢ ⎥
⎣1⎦

In [163]:
def round_expr(expr_list, num_digits):
    T = []
    for expr in expr_list:
        T.append(expr.xreplace({n : round(n, num_digits) for n in expr.atoms(Number)}))
    return T
T = round_expr(T,4)

In [236]:
f_1,f_2,f_3 = symbols('f1:4')
fs = Matrix([[f_1],[f_2],[f_3],[1]])

f = T[2]*T[3][:,[3]]
f1 = f[0,0]
f2 = f[1,0]
f3 = f[2,0]

In [237]:
g_1,g_2,g_3 = symbols('g1:4')
gs = Matrix([[g_1],[g_2],[g_3],[1]])

g = T[1]*fs
g1 = g[0,0]
g2 = g[1,0]
g3 = g[2,0]

In [238]:
P = simplify(T[0]*gs)
r = simplify(P[0,0]**2 + P[1,0]**2 + P[2,0]**2)
z = P[2,0]

In [239]:
P = simplify(T[0]*T[1]*T[2]*T[3][:,[3]])

In [240]:
r = simplify(P[0,0]**2 + P[1,0]**2 + P[2,0]**2)
z = P[2,0]

In [241]:
P

⎡-0.01822⋅sin(θ₃)⋅cos(θ₁ + θ₂) - 0.01818⋅sin(θ₁ + θ₂)⋅cos(θ₃) - 0.05⋅sin(θ₁ + θ₂)⎤
⎢                                                                                ⎥
⎢-0.01822⋅sin(θ₃)⋅sin(θ₁ + θ₂) + 0.01818⋅cos(θ₃)⋅cos(θ₁ + θ₂) + 0.05⋅cos(θ₁ + θ₂)⎥
⎢                                                                                ⎥
⎢                           0.0009838⋅cos(θ₃) - 1.263                            ⎥
⎢                                                                                ⎥
⎣                                       1                                        ⎦

In [242]:
r

            2                                
3.614e-7⋅sin (θ₃) - 0.0006694⋅cos(θ₃) + 1.599

In [243]:
z

0.0009838⋅cos(θ₃) - 1.263

In [244]:
solution = solve(0-z,thetas[2])
solution

[7.85096652027279⋅ⅈ, 6.28318530717959 - 7.85096652027279⋅ⅈ]

In [245]:
theta3 = [sol.evalf() for sol in solution if sol.is_real ]

In [229]:
theta3

[]